In [ ]:
import numpy as np

class ReplayBuffer:
    def __init__(self, max_size, obs_dim, act_dim):
        self.max_size = max_size
        self.ptr = 0
        self.size = 0
        
        self.obs_buf = np.zeros((max_size, obs_dim), dtype=np.float32)
        self.next_obs_buf = np.zeros((max_size, obs_dim), dtype=np.float32)
        self.act_buf = np.zeros((max_size, act_dim), dtype=np.float32)
        self.rew_buf = np.zeros((max_size, 1), np.float32)
        
    def store(self, obs, act, rew, next_obs, done):
        self.obs_buf[self.ptr] = obs
        self.act_buf[self.ptr] = act
        self.rew_buf[self.ptr] = rew
        self.next_obs_buf[self.ptr] = next_obs
        
        self.ptr = (self.ptr + 1) % self.max_size
        self.size = min(self.size + 1, self.max_size)
        
    def sample_batch(self, batch_size=32):
        idxs = np.random.randint(0, self.max_size, size=batch_size)
        
        return dict(obs = self.obs_buf[idxs],
                    act = self.act_buf[idxs],
                    rews = self.rew_buf[idxs],
                    next_obs = self.next_obs_buf[idxs])

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class ActorCritic(keras.Model):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        
        self.actor = keras.Sequential([
            layers.Dense(256, activation='relu'),
            layers.LayerNormalization(),
            layers.Dense(256, activation='relu'),
            layers.Dense(act_dim, activation='tanh')
        ])
        
        self.critic = keras.Sequential([
            layers.Dense(256, activation='relu'),
            layers.LayerNormalization(),
            layers.Dense(256, activation='relu'),
            layers.Dense(1)
        ])
        
    def call(self, inputs):
        obs = inputs
        return self.actor(obs), self.critic(obs)

In [ ]:
class Agent:
    def __init__(self, obs_dim, act_dim):
        self.model = ActorCritic(obs_dim, act_dim)
        self.target_model = ActorCritic(obs_dim, act_dim)
        self.target_model.set_weights(self.model.get_weights())
        
        self.buffer = ReplayBuffer(max_size=100000, obs_dim=obs_dim, act_dim=act_dim)
        self.gamma = 0.99
        self.batch_size = 64
        self.optimizer = keras.optimizers.Adam(learning_rate=3e-4)
        
    def act(self, obs):
        obs = tf.convert_to_tensor(obs.reshape(1, -1), dtype=np.float32)
        action, _ = self.model(obs)
        return action.numpy()[0]
    
    def learn(self):
        if self.batch_size > self.buffer.size:
            return
        
        batch = self.buffer.sample_batch(self.batch_size)
        
        obs = tf.convert_to_tensor(batch['obs'], dtype=np.float32)
        act = tf.convert_to_tensor(batch['act'], dtype=np.float32)
        rews = tf.convert_to_tensor(batch['rews'], dtype=np.float32)
        next_obs = tf.convert_to_tensor(batch['next_obs'], dtype=np.float32)
        
        with tf.GradientTape() as tape:
            _, value = self.model(obs)
            _, next_value = self.target_model(next_obs)
            target = rews + self.gamma*(1-done)*next_value
            critic_loss = tf.reduce_mean((value - target)**2)
            
        gradient = tape.gradient(critic_loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradient, self.model.trainable_variables))
        
        tau = 0.005
        new_weights = []
        target_variables = self.target_model.weights
        for i, variable in enumerate(self.model.weights):
            new_weights.append(tau * variable + (1 - tau) * target_variables[i])
        self.target_model.set_weights(new_weights)

In [ ]:
class env:
    def step(act):
        ~~
        line = arduino.readline().decode('utf-8').strip()
        if line.startswith("Accel:"):
            parts = line.split('||')
            accel_str = parts[0].split(':')[1]
            gyro_str = parts[1].split(':')[1]

            accel = [float(x) for x in accel_str.split(',')]
            gyro = [float(x) for x in gyro_str.split(',')]

            obs_ = np.array(accel + gyro, dtype=np.float32)
            
        reward = abs(obs_ - obs)
            
        return obs_, reward

In [ ]:
import serial
import time

arduino = serial.Serial("COM7", 9600)
time.sleep(2)

act_dim = 2
obs_dim = 6

agent = Agent(obs_dim, act_dim)

num_episodes = 10

line = arduino.readline().decode('utf-8').strip()
if line.startswith("Accel:"):
    parts = line.split('||')
    accel_str = parts[0].split(':')[1]
    gyro_str = parts[1].split(':')[1]

    accel = [float(x) for x in accel_str.split(',')]
    gyro = [float(x) for x in gyro_str.split(',')]

    obs = np.array(accel + gyro, dtype=np.float32)

for ep in range(num_episodes):
    
    if arduino.in_waiting > 0:
        try:    
            total_reward = 0
            waiting_time = 0
            
            for t in range(1000): # 5000frame --> step*5 = frame
                action = agent.act(obs)
                noise_scale = max(0.1, 1.0 - ep / num_episodes)  # 점점 줄이기
                noise = np.random.normal(-180, noise_scale, size=action.shape)
                action += noise

                action = np.clip(action, 0, 180)
                next_obs, reward = env.step(action)
                
                agent.buffer.store(obs, action, reward, next_obs)
                agent.learn()
                
                obs = next_obs
                total_reward += shaped_reward

                
            print(f"episode: {ep}, total reward: {total_reward}")
            rew_step.append(total_reward)

input()
env.close()